# 

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list, system_prompt
import selfies as sf

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_path = '/data/data/BioT5_chebi20/BioT5_chebi20_train.csv'
test_path = '/data/data/BioT5_chebi20/BioT5_chebi20_test.csv'

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

In [3]:
train_data.columns

Index(['CID', 'SMILES', 'description', 'polararea', 'xlogp', 'inchi',
       'iupacname', 'SELFIES'],
      dtype='object')

In [3]:

train_selfies = train_data['SELFIES'].tolist()
train_texts = train_data['description'].tolist()
train_smiles = [sf.decoder(i) for i in train_selfies]
train_mols = [Chem.MolFromSmiles(i) for i in train_smiles]


test_selfies = test_data['SELFIES'].tolist()
test_texts = test_data['description'].tolist()
test_smiles = [sf.decoder(i) for i in test_selfies]
test_mols = [Chem.MolFromSmiles(i) for i in test_smiles]

[07:44:22] WARNING: not removing hydrogen atom without neighbors
[07:44:22] WARNING: not removing hydrogen atom without neighbors
[07:44:22] WARNING: not removing hydrogen atom without neighbors
[07:44:22] WARNING: not removing hydrogen atom without neighbors
[07:44:23] WARNING: not removing hydrogen atom without neighbors
[07:44:23] WARNING: not removing hydrogen atom without neighbors
[07:44:23] WARNING: not removing hydrogen atom without neighbors
[07:44:23] WARNING: not removing hydrogen atom without neighbors
[07:44:23] WARNING: not removing hydrogen atom without neighbors
[07:44:23] WARNING: not removing hydrogen atom without neighbors
[07:44:23] WARNING: not removing hydrogen atom without neighbors
[07:44:23] WARNING: not removing hydrogen atom without neighbors
[07:44:24] WARNING: not removing hydrogen atom without neighbors
[07:44:24] WARNING: not removing hydrogen atom without neighbors
[07:44:24] WARNING: not removing hydrogen atom without neighbors
[07:44:24] WARNING: not r

In [5]:
task = 'chebi-20-mol2text'
list_train_data = get_data_list(
    list_mol=train_mols,
    list_label=train_texts,
    task=task,
    instruction_templates=instructions_smol.molecule_captioning,
)

list_test_data = get_data_list(
    list_mol=test_mols,
    list_label=test_texts,
    task=task,
    instruction_templates=instructions_smol.molecule_captioning,
)

 15%|█▍        | 3947/26407 [00:05<00:33, 661.11it/s]


KeyboardInterrupt: 

In [6]:
data_dict = {
    "train": list_train_data,
    "test": list_test_data
}


for split in ["train", "test"]:
    list_data = data_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_{task}_0219"
    )

Saving the dataset (1/1 shards): 100%|██████████| 3300/3300 [00:00<00:00, 8312.94 examples/s]


In [8]:
train_texts[0]

'The molecule is an epoxy(hydroxy)icosatrienoate that is the conjugate base of 11 hydroxy-(14R,15S)-epoxy-(5Z,8Z,12E)-icosatrienoic acid, obtained by deprotonation of the carboxy group; major species at pH 7.3. It is a conjugate base of an 11 hydroxy-(14R,15S)-epoxy-(5Z,8Z,12E)-icosatrienoic acid.'

In [9]:
instructions_smol.molecule_generation

['Based on the given information, generate a molecule that meets the desired specifications: <INPUT>',
 'Give me a molecule that satisfies the conditions outlined in the description: <INPUT>',
 'Generate a molecule based on this description: <INPUT>',
 'Can you create a molecule that matches the given characteristics? <INPUT>',
 'I need a molecule that meets the following conditions: <INPUT> Please represent the molecule in SELFIES.',
 'Suppose there is a molecule that meets the following description: <INPUT> Please write the SELFIES representation of it.',
 '<INPUT> Use the above information to create a molecule.',
 'Build a molecule that meets the requirement: <INPUT>',
 'Generate a molecule that fulfills the requirement: <INPUT>',
 'Conceptualize a molecule that meets the specified attribute(s): <INPUT>',
 'Come up with a molecule based on the description: <INPUT>',
 'Could you please return a molecule that adheres to this description? <INPUT>',
 'I give you a description of a molec

In [4]:
task = 'chebi-20-text2mol'
list_train_data = get_data_list(
    list_mol=train_texts,
    list_label=train_selfies,
    task=task,
    instruction_templates=instructions_smol.molecule_generation,
)

list_test_data = get_data_list(
    list_mol=test_texts,
    list_label=test_selfies,
    task=task,
    instruction_templates=instructions_smol.molecule_generation,
)

100%|██████████| 3300/3300 [00:00<00:00, 7028.20it/s]


In [5]:
list_test_data[0]

{'task': 'chebi-20-text2mol',
 'x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 0, 4, 5, 2, 0, 2, 0, 0],
        [5, 0, 4, 5, 3, 0, 2, 0, 0]]),
 'edge_index': array([[0, 1, 1, 2],
        [1, 0, 2, 1]]),
 'edge_attr': array([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]),
 'additional_x': array([[5, 0, 4, 5, 3, 0, 2, 0, 0],
        [5, 0, 4, 5, 2, 0, 2, 0, 0],
        [5, 0, 4, 5, 3, 0, 2, 0, 0]]),
 'additional_edge_index': array([[0, 1, 1, 2],
        [1, 0, 2, 1]]),
 'additional_edge_attr': array([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]),
 'input_mol_string': 'The molecule is a steroid ester that is methyl (17E)-pregna-4,17-dien-21-oate substituted by oxo groups at positions 3 and 11. It is a 3-oxo-Delta(4) steroid, an 11-oxo steroid, a steroid ester and a methyl ester. It derives from a hydride of a pregnane.',
 'prompt_text': '<s>[INST] You are a helpful assistant for molecular chemistry, to address tasks including molecular pro

In [6]:
data_dict = {
    "train": list_train_data,
    "test": list_test_data
}


for split in ["train", "test"]:
    list_data = data_dict[split]
    
    dataset = datasets.Dataset.from_list(list_data)
    dataset.save_to_disk(
        f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_{task}_0219"
    )

Saving the dataset (1/1 shards): 100%|██████████| 3300/3300 [00:00<00:00, 39193.55 examples/s]
